# Lesson 5: Valuing the Portfolio - Aggregations & GroupBy

The investment team has landed in your office with a single urgent question:  
**"What is this portfolio actually worth?"**

They want to know:
- How many properties do we have in each neighborhood?
- What is the average asking price by property type?
- Which neighborhoods dominate the market by total value?
- How does the mix of property types vary across neighborhoods?

Raw rows cannot answer these questions. You need **aggregations** - the art of collapsing many rows into meaningful summaries.

In this lesson you will learn:
- `groupBy()` + `agg()` to summarize data by category
- Built-in aggregate functions: `avg`, `sum`, `min`, `max`, `count`, `countDistinct`
- Filtering aggregated results (the SQL `HAVING` equivalent)
- Pivot tables to build a market matrix
- Computing percentage shares of total market value


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, sum, min, max, count, countDistinct, round
)

spark = SparkSession.builder \
    .appName("Lesson5_ValuingThePortfolio") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Load the property listings
listings_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/property_listings.csv")

# Drop rows that are missing the columns we need for aggregation
listings = listings_raw.dropna(subset=["list_price", "sqft", "neighborhood"])

print(f"Raw row count : {listings_raw.count()}")
print(f"Clean row count: {listings.count()}")
listings.printSchema()

Raw row count : 510


Clean row count: 412
root
 |-- property_id: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- property_type: string (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft: double (nullable = true)
 |-- list_price: double (nullable = true)
 |-- year_built: double (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- status: string (nullable = true)
 |-- listing_date: date (nullable = true)
 |-- agent_id: string (nullable = true)



## Counting Properties - The Market Census

Before we calculate value, let's understand **volume**. How many properties are in each segment?

`groupBy()` splits the DataFrame into groups, and `count()` tells us how big each group is.  
Think of it as sorting all the listing sheets into labeled piles and then counting each pile.


In [2]:
# Total portfolio size
total_properties = listings.count()
print(f"Total properties in portfolio: {total_properties}")

# Count by property type - what kinds of properties do we have?
print("\n--- Properties by Type ---")
listings.groupBy("property_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

# Count by neighborhood - where are properties concentrated?
print("--- Properties by Neighborhood ---")
listings.groupBy("neighborhood") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

Total properties in portfolio: 412

--- Properties by Type ---


+-------------+-----+
|property_type|count|
+-------------+-----+
|        House|  110|
|        Condo|  106|
|    Townhouse|   90|
|    Apartment|   64|
|         NULL|   26|
|        Villa|   16|
+-------------+-----+

--- Properties by Neighborhood ---


+------------+-----+
|neighborhood|count|
+------------+-----+
|    Downtown|   64|
|     Seaside|   55|
|    Hillside|   55|
|  Greenfield|   54|
|   Riverside|   52|
|     Oakwood|   45|
|    Parkview|   44|
|     Midtown|   43|
+------------+-----+



## GroupBy + Agg - The Real Power

Counting is useful, but the investment team wants **prices**. This is where `agg()` shines.

The pattern is:
```python
df.groupBy("category_column").agg(
    avg("value_column"),
    min("value_column"),
    max("value_column")
)
```

You can pass **multiple aggregation functions** in a single `agg()` call.  
Each produces a new column in the result - no rows from the original data are kept.


In [3]:
# Price summary by neighborhood
print("--- Price Summary by Neighborhood (sorted by avg price desc) ---")
listings.groupBy("neighborhood").agg(
    round(avg("list_price"), 0).alias("avg_price"),
    min("list_price").alias("min_price"),
    max("list_price").alias("max_price"),
    count("property_id").alias("num_properties")
).orderBy(col("avg_price").desc()).show()

--- Price Summary by Neighborhood (sorted by avg price desc) ---


+------------+---------+---------+---------+--------------+
|neighborhood|avg_price|min_price|max_price|num_properties|
+------------+---------+---------+---------+--------------+
|   Riverside| 884968.0| 172499.0|1468321.0|            52|
|    Parkview| 839615.0| 160785.0|1460786.0|            44|
|     Seaside| 838448.0| 167118.0|1496419.0|            55|
|    Downtown| 831989.0| 196250.0|1496853.0|            64|
|     Oakwood| 822758.0| 182230.0|1490144.0|            45|
|     Midtown| 820162.0| 172490.0|1497758.0|            43|
|    Hillside| 797046.0| 151539.0|1489527.0|            55|
|  Greenfield| 756956.0| 165216.0|1497203.0|            54|
+------------+---------+---------+---------+--------------+



In [4]:
# Multiple column groupBy: neighborhood AND property type
# This is like asking: "for each neighborhood, break it down further by type"
print("--- Avg Price and Count by Neighborhood + Property Type ---")
listings.groupBy("neighborhood", "property_type").agg(
    round(avg("list_price"), 0).alias("avg_price"),
    count("property_id").alias("num_properties")
).orderBy("neighborhood", col("avg_price").desc()).show(40)

--- Avg Price and Count by Neighborhood + Property Type ---


+------------+-------------+---------+--------------+
|neighborhood|property_type|avg_price|num_properties|
+------------+-------------+---------+--------------+
|    Downtown|        Condo| 948475.0|            18|
|    Downtown|        House| 899764.0|            16|
|    Downtown|    Townhouse| 859826.0|            12|
|    Downtown|        Villa| 722276.0|             5|
|    Downtown|    Apartment| 641988.0|            10|
|    Downtown|         NULL| 476442.0|             3|
|  Greenfield|    Apartment| 868328.0|             8|
|  Greenfield|    Townhouse| 780861.0|            16|
|  Greenfield|        Condo| 774828.0|            12|
|  Greenfield|        House| 680079.0|            11|
|  Greenfield|         NULL| 669329.0|             5|
|  Greenfield|        Villa| 654877.0|             2|
|    Hillside|         NULL| 917459.0|             4|
|    Hillside|    Apartment| 875401.0|            10|
|    Hillside|    Townhouse| 829778.0|             9|
|    Hillside|        Condo|

## Filtering After Aggregation

In SQL you use `HAVING` to filter on aggregated values. In PySpark, you just chain a `.filter()` (or `.where()`) **after** the `.agg()` call.

The logic is identical: first aggregate, then apply the condition on the aggregated column.

```python
df.groupBy(...).agg(...).filter(col("agg_column") > threshold)
```

This is different from filtering **before** aggregation - here the filter applies to the summarized result, not individual rows.


In [5]:
# Find neighborhoods with more than 30 listings - significant markets only
print("--- Significant Neighborhoods (>30 listings) ---")
significant_neighborhoods = listings.groupBy("neighborhood").agg(
    count("property_id").alias("num_properties"),
    round(avg("list_price"), 0).alias("avg_price")
).filter(col("num_properties") > 30) \
 .orderBy(col("num_properties").desc())

significant_neighborhoods.show()
print(f"Neighborhoods with >30 listings: {significant_neighborhoods.count()}")

--- Significant Neighborhoods (>30 listings) ---


+------------+--------------+---------+
|neighborhood|num_properties|avg_price|
+------------+--------------+---------+
|    Downtown|            64| 831989.0|
|     Seaside|            55| 838448.0|
|    Hillside|            55| 797046.0|
|  Greenfield|            54| 756956.0|
|   Riverside|            52| 884968.0|
|     Oakwood|            45| 822758.0|
|    Parkview|            44| 839615.0|
|     Midtown|            43| 820162.0|
+------------+--------------+---------+



Neighborhoods with >30 listings: 8


## Pivot Tables - The Market Matrix

A pivot table rotates one column's values into new column headers. In real estate terms:  
instead of rows like `(Riverside, Condo, $450k)`, you get a row per neighborhood  
with columns `Condo`, `Single Family`, `Townhouse`, etc., each holding the average price.

This is the market matrix the investment team actually wants to present in their slides.

```python
df.groupBy("row_key").pivot("column_key").agg(avg("value"))
```


In [6]:
# Pivot: rows = neighborhood, columns = property_type, values = avg list_price
print("--- Market Matrix: Avg Price by Neighborhood x Property Type ---")
price_matrix = listings.groupBy("neighborhood") \
    .pivot("property_type") \
    .agg(round(avg("list_price"), 0))

price_matrix.show(truncate=False)

# The pivot creates one column per unique property_type value
print("Pivot result columns:", price_matrix.columns)

--- Market Matrix: Avg Price by Neighborhood x Property Type ---


+------------+--------+---------+--------+--------+---------+---------+
|neighborhood|null    |Apartment|Condo   |House   |Townhouse|Villa    |
+------------+--------+---------+--------+--------+---------+---------+
|Midtown     |651680.0|1115725.0|790912.0|825174.0|780167.0 |835799.0 |
|Seaside     |898075.0|792966.0 |689010.0|976996.0|809492.0 |797499.0 |
|Parkview    |816769.0|401656.0 |715934.0|938434.0|962249.0 |1311986.0|
|Downtown    |476442.0|641988.0 |948475.0|899764.0|859826.0 |722276.0 |
|Hillside    |917459.0|875401.0 |823943.0|588779.0|829778.0 |755108.0 |
|Greenfield  |669329.0|868328.0 |774828.0|680079.0|780861.0 |654877.0 |
|Riverside   |969497.0|655274.0 |918255.0|967175.0|882612.0 |1460015.0|
|Oakwood     |605433.0|674854.0 |904279.0|923378.0|779305.0 |1165691.0|
+------------+--------+---------+--------+--------+---------+---------+

Pivot result columns: ['neighborhood', 'null', 'Apartment', 'Condo', 'House', 'Townhouse', 'Villa']


## Advanced Aggregations

Two more functions are essential for portfolio analysis:

- `countDistinct()` - how many **unique** values exist? Useful for agent counts, unique zip codes, etc.
- `sum()` - total value. The sum of all list prices is the **total market value** of the portfolio.

These can be mixed freely inside a single `agg()` call alongside `avg`, `min`, `max`.


In [7]:
# Full portfolio summary in one agg() call
print("--- Portfolio Market Summary ---")
listings.agg(
    count("property_id").alias("total_listings"),
    countDistinct("agent_id").alias("unique_agents"),
    countDistinct("neighborhood").alias("neighborhoods_covered"),
    round(sum("list_price"), 0).alias("total_market_value"),
    round(avg("list_price"), 0).alias("avg_list_price"),
    round(avg("sqft"), 0).alias("avg_sqft"),
    min("list_price").alias("cheapest_listing"),
    max("list_price").alias("most_expensive_listing")
).show(vertical=True)

--- Portfolio Market Summary ---


-RECORD 0------------------------------
 total_listings         | 412          
 unique_agents          | 20           
 neighborhoods_covered  | 8            
 total_market_value     | 3.39327536E8 
 avg_list_price         | 823611.0     
 avg_sqft               | 2939.0       
 cheapest_listing       | 151539.0     
 most_expensive_listing | 1497758.0    



In [8]:
# Compute each neighborhood's share of total market value
# Step 1: get total market value as a scalar
total_value = listings.agg(sum("list_price").alias("total")).collect()[0]["total"]
print(f"Total portfolio market value: ${total_value:,.0f}")

# Step 2: aggregate by neighborhood, then compute percentage
print("\n--- Neighborhood Share of Total Market Value ---")
listings.groupBy("neighborhood").agg(
    count("property_id").alias("num_properties"),
    round(sum("list_price"), 0).alias("neighborhood_value")
).withColumn(
    "pct_of_market",
    round((col("neighborhood_value") / total_value) * 100, 2)
).orderBy(col("pct_of_market").desc()).show()

Total portfolio market value: $339,327,536

--- Neighborhood Share of Total Market Value ---


+------------+--------------+------------------+-------------+
|neighborhood|num_properties|neighborhood_value|pct_of_market|
+------------+--------------+------------------+-------------+
|    Downtown|            64|       5.3247279E7|        15.69|
|     Seaside|            55|       4.6114628E7|        13.59|
|   Riverside|            52|       4.6018333E7|        13.56|
|    Hillside|            55|       4.3837553E7|        12.92|
|  Greenfield|            54|       4.0875605E7|        12.05|
|     Oakwood|            45|       3.7024116E7|        10.91|
|    Parkview|            44|       3.6943062E7|        10.89|
|     Midtown|            43|        3.526696E7|        10.39|
+------------+--------------+------------------+-------------+



## Lesson 5 Wrap-Up

You now have the tools to answer "what is this portfolio worth?" at any level of granularity.

**What you learned:**
- `groupBy("col").count()` - count rows per group
- `groupBy("col").agg(avg(...), min(...), max(...), sum(...))` - multiple aggregations at once
- Multi-column `groupBy("col1", "col2")` - nested breakdowns
- `.filter()` after `.agg()` - the PySpark equivalent of SQL `HAVING`
- `.pivot()` - rotate a column into a cross-tab matrix
- `countDistinct()` and `sum()` for unique counts and totals
- Using a collected scalar to compute percentage shares

**Key mental model:** `groupBy` collapses many rows into one row per group.  
You lose the individual row detail - you gain the summary.

---

**Next: Lesson 6 - The Commission Report (Joins!)**  
The agents want their commission reports, but the commission data and property details live in separate files.  
Learn how to combine DataFrames with inner, left, right, full outer, anti, and broadcast joins.
